# Stage 1 — Data Ingestion

**Project:** Superstore retail profitability triage
**Business question:** *Which products, sub-categories, regions, and customer segments should
the Superstore grow, fix, or exit — and how much profit is recoverable from each decision?*

This notebook acquires the raw data, records its provenance, and validates its schema against a
declared data dictionary. It makes **no modifications** — raw data is immutable after ingestion
(`STRUCTURE.md` §1).

**Gate for this stage:** raw saved untouched · source/timestamp/counts logged · schema matches or
discrepancies documented · PII review completed.

In [1]:
from __future__ import annotations

import hashlib
import shutil
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE = ROOT / "Superstore.csv"
RAW = ROOT / "data" / "raw" / "Superstore.csv"
RAW.parent.mkdir(parents=True, exist_ok=True)

print(f"Project root : {ROOT.name}")
print(f"Source file  : {SOURCE.name} ({SOURCE.stat().st_size / 1_048_576:.2f} MB)")

Project root : Superstore Dataset
Source file  : Superstore.csv (2.18 MB)


## 1.1 Encoding — verify, don't assume

This file carries non-UTF8 bytes in customer/city name fields. A naive `read_csv` succeeds on a
small `nrows` peek and then **fails on the full read**, which is the worst possible failure mode:
it looks fine in exploration and breaks in the pipeline. We probe explicitly and pin the result.

In [2]:
def probe_encoding(path: Path, candidates=("utf-8", "latin-1", "cp1252")) -> str:
    """Return the first encoding that reads the file in full, not just a sample."""
    for enc in candidates:
        try:
            pd.read_csv(path, encoding=enc)
            print(f"  {enc:<10} full read OK")
            return enc
        except UnicodeDecodeError as exc:
            print(f"  {enc:<10} FAILED — {exc.reason} at byte {exc.start}")
    raise RuntimeError("No candidate encoding could read the file")


ENCODING = probe_encoding(SOURCE)
print(f"\nPinned encoding: {ENCODING!r}")

  utf-8      FAILED — invalid start byte at byte 9
  latin-1    full read OK

Pinned encoding: 'latin-1'


## 1.2 Load and record provenance

Metadata logging per `STRUCTURE.md` §1: source, pull timestamp, row/column counts, and a
SHA-256 hash so any future run can prove it read the same bytes.

In [3]:
def file_hash(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


raw = pd.read_csv(SOURCE, encoding=ENCODING)

lineage = {
    "source_file": SOURCE.name,
    "source_origin": "Tableau/Kaggle US Superstore sample (retail transactions)",
    "pull_timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    "encoding": ENCODING,
    "sha256": file_hash(SOURCE),
    "rows": len(raw),
    "columns": raw.shape[1],
    "size_mb": round(SOURCE.stat().st_size / 1_048_576, 2),
}

for k, v in lineage.items():
    print(f"{k:<20} {v}")

source_file          Superstore.csv
source_origin        Tableau/Kaggle US Superstore sample (retail transactions)
pull_timestamp_utc   2026-07-20 10:19:11
encoding             latin-1
sha256               58fb59c63089e564d835e4aabe19d371757de1137a5bf1888243ae386c4b7a97
rows                 9994
columns              21
size_mb              2.18


In [4]:
# Immutable copy. Everything downstream reads from data/raw/, never from the working file.
shutil.copy2(SOURCE, RAW)
print(f"Raw copy written to data/raw/{RAW.name} — this file is never edited again.")

Raw copy written to data/raw/Superstore.csv — this file is never edited again.


## 1.3 Data dictionary

Declared **before** looking at the data, so validation is a genuine test rather than a
description of whatever happened to load.

In [5]:
DATA_DICTIONARY = {
    "Row ID":        ("int",      "Surrogate row index — an artifact, not analytic. Dropped at Stage 2."),
    "Order ID":      ("str",      "Order key. Repeats across rows: one order holds many product lines."),
    "Order Date":    ("date",     "Date order was placed (M/D/YYYY)."),
    "Ship Date":     ("date",     "Date order shipped. Must be >= Order Date."),
    "Ship Mode":     ("category", "Standard / Second / First Class / Same Day."),
    "Customer ID":   ("str",      "Customer key."),
    "Customer Name": ("str",      "QUASI-PII — dropped at Stage 2; Customer ID covers all analysis."),
    "Segment":       ("category", "Consumer / Corporate / Home Office."),
    "Country":       ("str",      "Constant 'United States' — dropped at Stage 2 (zero variance)."),
    "City":          ("str",      "531 expected."),
    "State":         ("str",      "49 expected."),
    "Postal Code":   ("str",      "Cast to STRING — leading zeros are meaningful, it is not a number."),
    "Region":        ("category", "South / West / Central / East."),
    "Product ID":    ("str",      "Product key."),
    "Category":      ("category", "Furniture / Office Supplies / Technology."),
    "Sub-Category":  ("category", "17 expected."),
    "Product Name":  ("str",      "Free-text product description."),
    "Sales":         ("float",    "Line revenue, USD. Must be > 0."),
    "Quantity":      ("int",      "Units on the line. Must be >= 1."),
    "Discount":      ("float",    "Fraction 0-1. Discrete: 12 distinct values expected."),
    "Profit":        ("float",    "Line profit, USD. May be negative — losses are the finding."),
}

EXPECTED_DOMAINS = {
    "Region": {"South", "West", "Central", "East"},
    "Segment": {"Consumer", "Corporate", "Home Office"},
    "Category": {"Furniture", "Office Supplies", "Technology"},
    "Ship Mode": {"Standard Class", "Second Class", "First Class", "Same Day"},
}

EXPECTED_SHAPE = (9994, 21)
EXPECTED_CARDINALITY = {"Sub-Category": 17, "State": 49, "Customer ID": 793}

pd.DataFrame(
    [(c, t, d) for c, (t, d) in DATA_DICTIONARY.items()],
    columns=["column", "expected_type", "definition"],
)

,column,expected_type,definition
0,Row ID,int,"Surrogate row index — an artifact, not analyti..."
1,Order ID,str,Order key. Repeats across rows: one order hold...
2,Order Date,date,Date order was placed (M/D/YYYY).
3,Ship Date,date,Date order shipped. Must be >= Order Date.
4,Ship Mode,category,Standard / Second / First Class / Same Day.
5,Customer ID,str,Customer key.
6,Customer Name,str,QUASI-PII — dropped at Stage 2; Customer ID co...
7,Segment,category,Consumer / Corporate / Home Office.
8,Country,str,Constant 'United States' — dropped at Stage 2 ...
9,City,str,531 expected.


## 1.4 Schema validation

Assertions, not eyeballing. A discrepancy here must be documented before the pipeline advances.

In [6]:
checks = []


def check(name: str, passed: bool, detail: str = "") -> None:
    checks.append({"check": name, "result": "PASS" if passed else "FAIL", "detail": detail})


check("Shape matches expectation", raw.shape == EXPECTED_SHAPE, f"{raw.shape} vs {EXPECTED_SHAPE}")
check(
    "All dictionary columns present",
    set(DATA_DICTIONARY) == set(raw.columns),
    f"missing={set(DATA_DICTIONARY) - set(raw.columns)} extra={set(raw.columns) - set(DATA_DICTIONARY)}",
)
check("Row ID is a unique key", raw["Row ID"].is_unique, f"{raw['Row ID'].nunique():,} unique")

for col, expected in EXPECTED_DOMAINS.items():
    actual = set(raw[col].unique())
    check(f"{col} domain is closed", actual == expected, f"unexpected={actual - expected or 'none'}")

for col, n in EXPECTED_CARDINALITY.items():
    check(f"{col} cardinality = {n}", raw[col].nunique() == n, f"actual={raw[col].nunique()}")

check("Country is constant", raw["Country"].nunique() == 1, f"values={raw['Country'].unique()}")
check("Sales strictly positive", bool((raw["Sales"] > 0).all()), f"min={raw['Sales'].min():.2f}")
check("Quantity >= 1", bool((raw["Quantity"] >= 1).all()), f"min={raw['Quantity'].min()}")
check("Discount within [0, 1]", bool(raw["Discount"].between(0, 1).all()),
      f"range=[{raw['Discount'].min()}, {raw['Discount'].max()}]")

schema_report = pd.DataFrame(checks)
print(f"{(schema_report['result'] == 'PASS').sum()}/{len(schema_report)} checks passed\n")
schema_report

14/14 checks passed



,check,result,detail
0,Shape matches expectation,PASS,"(9994, 21) vs (9994, 21)"
1,All dictionary columns present,PASS,missing=set() extra=set()
2,Row ID is a unique key,PASS,"9,994 unique"
3,Region domain is closed,PASS,unexpected=none
4,Segment domain is closed,PASS,unexpected=none
5,Category domain is closed,PASS,unexpected=none
6,Ship Mode domain is closed,PASS,unexpected=none
7,Sub-Category cardinality = 17,PASS,actual=17
8,State cardinality = 49,PASS,actual=49
9,Customer ID cardinality = 793,PASS,actual=793


In [7]:
assert (schema_report["result"] == "PASS").all(), "Schema validation failed — resolve before Stage 2."
print("Schema validation clean. Cleared to proceed to Stage 2.")

Schema validation clean. Cleared to proceed to Stage 2.


## 1.5 Grain and coverage

**Grain is the single most important thing to state at ingestion** (`STRUCTURE.md`
§Modality-Discipline-Rules). Getting this wrong silently corrupts every downstream aggregation.

In [8]:
dates = pd.to_datetime(raw["Order Date"], format="%m/%d/%Y")

print(f"Grain            : one row per PRODUCT LINE within an order")
print(f"                   {len(raw):,} lines across {raw['Order ID'].nunique():,} orders "
      f"({len(raw) / raw['Order ID'].nunique():.2f} lines/order)")
print(f"Date coverage    : {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d} "
      f"({(dates.max() - dates.min()).days / 365.25:.1f} years)")
print(f"Customers        : {raw['Customer ID'].nunique():,}")
print(f"Geography        : {raw['Country'].iloc[0]} — {raw['Region'].nunique()} regions, "
      f"{raw['State'].nunique()} states, {raw['City'].nunique()} cities")
print(f"Assortment       : {raw['Category'].nunique()} categories, "
      f"{raw['Sub-Category'].nunique()} sub-categories, {raw['Product ID'].nunique():,} products")
print()
print("NOTE: order-line grain means a 'customer' or 'order' metric requires explicit aggregation.")
print("      Averaging a line-level margin across lines is NOT the same as order-level margin.")

Grain            : one row per PRODUCT LINE within an order
                   9,994 lines across 5,009 orders (2.00 lines/order)
Date coverage    : 2014-01-03 to 2017-12-30 (4.0 years)
Customers        : 793
Geography        : United States — 4 regions, 49 states, 531 cities
Assortment       : 3 categories, 17 sub-categories, 1,862 products

NOTE: order-line grain means a 'customer' or 'order' metric requires explicit aggregation.
      Averaging a line-level margin across lines is NOT the same as order-level margin.


## 1.6 Governance & PII review

Required at Stage 1 before data enters the local environment (`STRUCTURE.md` §Cross-Cutting).

In [9]:
print("PII classification")
print("-" * 78)
print("Customer Name  QUASI-IDENTIFIER. Real personal names.")
print("               -> Not required for any analysis in this project: Customer ID supports")
print("                  every customer-level feature (order count, recency) without it.")
print("               -> DECISION: dropped at Stage 2 (data minimisation).")
print()
print("Postal Code /  Geographic, but aggregated to city level at minimum and never joined to")
print("City / State   an external identity source. Retained — regional analysis is the point.")
print()
print("No payment, contact, demographic, or protected-attribute fields are present.")
print("No model in this project scores people; the classifier scores product-order lines.")
print("Consequently a fairness audit is NOT applicable — stated explicitly, not silently skipped.")

PII classification
------------------------------------------------------------------------------
Customer Name  QUASI-IDENTIFIER. Real personal names.
               -> Not required for any analysis in this project: Customer ID supports
                  every customer-level feature (order count, recency) without it.
               -> DECISION: dropped at Stage 2 (data minimisation).

Postal Code /  Geographic, but aggregated to city level at minimum and never joined to
City / State   an external identity source. Retained — regional analysis is the point.

No payment, contact, demographic, or protected-attribute fields are present.
No model in this project scores people; the classifier scores product-order lines.
Consequently a fairness audit is NOT applicable — stated explicitly, not silently skipped.


## 1.7 First look

Structure only. Interpretation belongs in Stage 3, not here.

In [10]:
print(raw.dtypes.to_string())
print()
print(f"Nulls across all columns : {raw.isnull().sum().sum()}")
print(f"Exact duplicate rows     : {raw.duplicated().sum()}")
raw.head()

Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64

Nulls across all columns : 0
Exact duplicate rows     : 0


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


> **Note on data quality.** Zero nulls and zero duplicates is *not* what real operational data
> looks like. This is a pre-scrubbed teaching dataset. The consequence is that Stage 2 is
> thinner than a real engagement would demand — the cleaning work here is type discipline and
> feature derivation, not repair. This is carried into the report's limitations rather than
> presented as a pipeline that survived a stress test.

In [11]:
lineage_df = pd.DataFrame([lineage]).T.rename(columns={0: "value"})
lineage_df.to_csv(ROOT / "data" / "raw" / "_lineage.csv")
schema_report.to_csv(ROOT / "data" / "raw" / "_schema_validation.csv", index=False)
print("Lineage and schema-validation logs written to data/raw/.")

Lineage and schema-validation logs written to data/raw/.


## Stage 1 — Gate Checklist

- [x] Raw data saved to `data/raw/` (untouched copy)
- [x] Source, timestamp, encoding, SHA-256, and row/column counts logged
- [x] Schema matches expectations — 13/13 assertions pass
- [x] Grain declared explicitly (order-line, not order or customer)
- [x] PII review completed — `Customer Name` flagged for removal at Stage 2

**Next:** `02_cleaning.ipynb` — type casting, business-rule validation, derived columns.